In [ ]:
import numpy as np
from scipy.special import i0, expit
from scipy.stats import vonmises
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

from src.model import jammalamadaka_sharma_corr, angular_absolute_error, sos_swq_tmin, sos_swq_tmin_predictor, sos_swq_tmin_duration

meta = pd.read_parquet("phenocam_site_meta.parquet")
pheno = pd.read_parquet("phenocam_phenology_model.parquet")
climate = pd.read_parquet("phenocam_site_in_usa_climate.parquet")
climate.columns = ["year", "yday", "prcp", "tmax", "tmin", "vp", "site"]

In [ ]:
meta

### Prepare data

In [ ]:
# Reshape long-format climate data to one row per site-year,
# with array columns containing the full daily time series.
climate_arrays = (
    climate
    .sort_values(["site", "year", "yday"])
    .groupby(["site", "year"], as_index=False)
    .apply(lambda g: pd.Series({
        "tmax": g["tmax"].values,
        "prcp": g["prcp"].values
    }))
)

print(climate_arrays.shape)
climate_arrays.head()

In [ ]:
# Select only the phenology columns we need and join to climate_arrays.
pheno_subset = pheno[["site", "transition_year_rising", "veg_type", "doy_transition_50_rising"]].copy()

model_data = climate_arrays.merge(
    pheno_subset,
    left_on=["site", "year"],
    right_on=["site", "transition_year_rising"],
    how="inner"
)

# Drop the redundant year column from the phenology table
model_data = model_data.drop(columns=["transition_year_rising"])

print(model_data.shape)
model_data.head()

### Model definition

See `src.model` for details.

In [ ]:
# From Table 3 in
# https://www.sciencedirect.com/science/article/pii/S0168192322003914#tbl0003
b      = 3.6
c      = 1.38
Preq   = 20.17
Tthres = -10.14
Tb     = 4.03
Fcrit  = 192

temp = model_data.tmax.iloc[0]
prcp = model_data.prcp.iloc[0]

P_sigmoid, Sw, k, T_forcing, Sf = sos_swq_tmin(
    temp, prcp,
    b, c, Preq, Tthres, Tb, Fcrit
)

Sf = np.minimum(Fcrit, Sf)

titles = (
    "Daily precip forcing",
    "Cumulative precip forcing",
    #"Precip threshold",
    "Daily temp forcing",
    "Cumulative temp forcing"
)

sos_doy = np.argmax(Sf >= Fcrit)
prcp_doy = np.argmax(Sw >= Preq)

fig, ax = plt.subplots(2, 2, figsize=(8, 5))

for axis, data, title in zip(ax.flat, (P_sigmoid, Sw, T_forcing, Sf), titles):
    axis.plot(data)
    axis.set_title(title, loc="left")

ax.flat[-1].plot(sos_doy, Sf[sos_doy], marker="*", color="red")
for a in ax.flat:
    a.axvline(prcp_doy, color="black", linestyle="dashed")
    
ax.flat[1].plot(prcp_doy, Sw[prcp_doy], marker="*", color="red")
#ax[-1].set_yscale("log")

fig.supxlabel("Day of year")

plt.tight_layout()
plt.savefig("figures/post_prcp_tmax_model.png")
plt.show()

In [ ]:
from scipy.optimize import minimize

DOY_MAX = 365

def doy_to_radians(doy, doy_max=DOY_MAX):
    """Convert day-of-year to radians on [0, 2*pi)."""
    return 2 * np.pi * (doy - 1) / doy_max

def pheno_model_error_function(params, observed_doy, tmax_arrays, prcp_arrays, doy_max=DOY_MAX):
    """Error function for phenology models."""
    sos_doy = sos_swq_tmin_predictor(tmax_arrays, prcp_arrays, *params)+1
    sum_sq_error = np.sum(angular_absolute_error(observed_doy, sos_doy, cycle_length=365) ** 2)
    return sum_sq_error

In [ ]:
from tqdm.autonotebook import tqdm

tmax_arrays = np.stack(model_data["tmax"])
prcp_arrays = np.stack(model_data["prcp"])

# Bounds loosely based on parameter ranges in Table 3
bounds = [
    (0, 200), 
    (0, 20), 
    (0, 100), 
    (-20, 10), 
    (-20, 10), 
    (0, 3000)
]

def fit_sos_swq_tmin_model(y_obs, tmax_arrays, prcp_arrays, bounds, n_starts=1000):
    best = None
    n_starts = 1000
    
    for x0 in tqdm(range(n_starts)):
        # Randomly draw a starting spot in the parameter space
        x0 = np.random.uniform(
            low  = [b[0] for b in bounds],
            high = [b[1] for b in bounds]
        )
        
        res = minimize(
            pheno_model_error_function,
            args=(
                y_obs,
                tmax_arrays,
                prcp_arrays
            ),
            x0=x0,
            method="L-BFGS-B",
            bounds=bounds,
        )
        
        if best is None or res.fun < best.fun:
            best = res
    
    return best

global_fit = fit_sos_swq_tmin_model(model_data["doy_transition_50_rising"].values, tmax_arrays, prcp_arrays, bounds)

In [ ]:
global_yhat = sos_swq_tmin_predictor(tmax_arrays, prcp_arrays, *global_fit.x)+1

In [ ]:
jammalamadaka_sharma_corr(model_data["doy_transition_50_rising"], global_yhat, cycle_length=365)

In [ ]:
angular_absolute_error(model_data["doy_transition_50_rising"], global_yhat).mean()

In [ ]:
plt.scatter(model_data["doy_transition_50_rising"]%DOY_MAX, global_yhat)
plt.axline((0, 0), (1, 1), color="black", linestyle="dashed")
plt.grid()
plt.gca().set_aspect("equal")

To fit by each veg type.

In [ ]:
results_veg = []

params_veg = {}

for grp, subdf in model_data.groupby("veg_type"):
    if subdf.shape[0] > 100:
        model_subdf = subdf.copy()
    else:
        params_veg[grp] = global_fit
        continue
        
    tmax_arrays = np.stack(model_subdf["tmax"])
    prcp_arrays = np.stack(model_subdf["prcp"])
    
    # Keep initial guess and bounds from global model
    group_fit = fit_sos_swq_tmin_model(
        model_subdf["doy_transition_50_rising"].values, tmax_arrays, prcp_arrays, bounds
    )

    model_subdf["model_doy_transition_50_rising"] = sos_swq_tmin_predictor(tmax_arrays, prcp_arrays, *group_fit.x)+1

    params_veg[grp] = group_fit
    
    results_veg.append(model_subdf[model_subdf.veg_type == grp])

veg_type_results = pd.concat(results_veg, axis=0)

To fit by veg type and climate classification

In [ ]:
model_data["koeppen"] = meta["koeppen_geiger"].loc[model_data["site"]].to_numpy()
model_data["koeppen1"] = model_data["koeppen"].str[0:1]
model_data["koeppen2"] = model_data["koeppen"].str[0:2]

model_data.groupby("koeppen2").count()

In [ ]:
results_kop = []

params_kop = {}

for grp, subdf in model_data.groupby("koeppen2"):
    if subdf.shape[0] > 50:
        model_subdf = subdf.copy()
    else:
        params_kop[grp] = global_fit
        continue
        
    tmax_arrays = np.stack(model_subdf["tmax"])
    prcp_arrays = np.stack(model_subdf["prcp"])
    
    # Keep initial guess and bounds from global model
    group_fit = fit_sos_swq_tmin_model(
        model_subdf["doy_transition_50_rising"].values, tmax_arrays, prcp_arrays, bounds
    )

    model_subdf["model_doy_transition_50_rising"] = sos_swq_tmin_predictor(tmax_arrays, prcp_arrays, *group_fit.x)+1

    params_kop[grp] = group_fit
    
    results_kop.append(model_subdf)

kop_type_results = pd.concat(results_kop, axis=0)

In [ ]:
kop_type_results

In [ ]:
from matplotlib import pyplot as plt

values, categories = model_data["veg_type"].factorize()
palette = sns.color_palette("Set2", len(categories))
palette_dict = dict(zip(categories, palette))

fig, axes = plt.subplots(1, 4, sharex=True, sharey=True, figsize=(10, 4))

ax1, ax2, ax3, ax4 = axes.flat

sns.scatterplot(ax=ax1, x=model_data["doy_transition_50_rising"] % DOY_MAX, y=global_yhat, hue=model_data["veg_type"], palette=palette_dict, s=(5)**2)
sns.scatterplot(ax=ax2, x=veg_type_results["doy_transition_50_rising"] % DOY_MAX, y=veg_type_results["model_doy_transition_50_rising"], hue=veg_type_results["veg_type"], palette=palette_dict, s=5**2)
sns.scatterplot(ax=ax3, x=kop_type_results["doy_transition_50_rising"] % DOY_MAX, y=kop_type_results["model_doy_transition_50_rising"], hue=veg_type_results["veg_type"], palette=palette_dict, s=5**2)

#ax1.scatter(model_data["doy_transition_50_rising"] % DOY_MAX, global_yhat, s=1)
#ax2.scatter(veg_type_results["doy_transition_50_rising"] % DOY_MAX, veg_type_results["model_doy_transition_50_rising"], s=1)

ax1.get_legend().remove()
ax2.get_legend().remove()
ax3.get_legend().remove()
    
handles, labels = ax1.get_legend_handles_labels()
ax4.legend(handles, labels, loc="center")
ax4.axis("off")

ax1.set_title("Global")
ax2.set_title("By veg type")
ax3.set_title("By climate class")

for a in (ax1, ax2, ax3):
    a.set_xlim(-10, 375)
    a.set_ylim(-10, 375)
    a.set_aspect("equal")
    a.grid()
    a.axline((0, 0), (1, 1), color="black")
    a.set_xlabel("")
    a.set_ylabel("")
    
fig.supxlabel("Observed greenup DOY")
fig.supylabel("Model greenup DOY")

plt.tight_layout()
plt.savefig("figures/forcing_model_one_to_one.png")
plt.show()

Performance metrics

In [ ]:
def print_performance(y_obs, y_hat):
    print(f"JS-score: {jammalamadaka_sharma_corr(y_obs, y_hat):.2f}")
    print(f"AAE: {angular_absolute_error(y_obs, y_hat).mean():.2f}")
    print(f"ARMSE: {np.sqrt((angular_absolute_error(y_obs, y_hat) ** 2).mean()):.2f}")

print("Global model")
print_performance(model_data["doy_transition_50_rising"], global_yhat)

print("By veg type")
print_performance(veg_type_results["doy_transition_50_rising"], veg_type_results["model_doy_transition_50_rising"])

print("By climate class")
print_performance(kop_type_results["doy_transition_50_rising"], kop_type_results["model_doy_transition_50_rising"])

### Persist parameters

In [ ]:
import inspect
parnames = list(inspect.signature(sos_swq_tmin).parameters.keys())[2:] # drop tmin/prcp

In [ ]:
veg_param_table = pd.DataFrame({key: val.x for key, val in params_veg.items()}, index=parnames)
kop_param_table = pd.DataFrame({key: val.x for key, val in params_kop.items()}, index=parnames)

veg_param_table.to_parquet("phenocam_forcing_model_parameters_veg_type.parquet")
kop_param_table.to_parquet("phenocam_forcing_model_parameters_kop_type.parquet")

In [ ]:
param_table = pd.DataFrame(params, index=parnames)
param_table.to_parquet("phenocam_forcing_model_parameters.parquet")

In [ ]:
# Check how we would do inference across multiple veg types. This is recreating the above groupby
# loop in one shot by broadcasting inputs to greenup_activity.
tmax_arrays = np.stack(veg_type_results["tmax"])
prcp_arrays = np.stack(veg_type_results["prcp"])
veg_type    = veg_type_results["veg_type"]

param_by_veg = param_table.T.loc[veg_type]

In [ ]:
est_greenup = greenup_activity(
    tmax_arrays, prcp_arrays,
    param_by_veg[["tmin"]].to_numpy(), param_by_veg[["temp_sigmoid_inflection"]].to_numpy(), param_by_veg[["temp_sigmoid_slope"]].to_numpy(),
    param_by_veg[["prcp_sigmoid_inflection"]].to_numpy(), param_by_veg[["prcp_sigmoid_slope"]].to_numpy(),
    0.5
)
est_greenup_doy = np.argmax(est_greenup >= 0.5, axis=1) + 1 # account for zero-based indexing
est_greenup_doy[est_greenup_doy == 0] = 365

In [ ]:
assert np.allclose(est_greenup_doy, veg_type_results["model_doy_transition_50_rising"])